# 🏔️ LITHOS Phase 9 — Full North East Coverage (30m, Copernicus DEM)
Gap-free slope-unit segmentation for all 8 NE states using **COPERNICUS/DEM/GLO30** (via Google Earth Engine) and **WhiteboxTools** hillslope segmentation.

**Runtime:** CPU is fine (no GPU needed for this notebook).

**Before you run this:**
1. You need a Google Earth Engine account with a Cloud project enabled for the EE API. Put your project id in `GEE_PROJECT` below.
2. Your Google Drive must contain (or will contain) `My Drive/LITHOS/Phase9_data/`.
3. If you already have a baseline file `lithos_all_slope_units_final.gpkg` in that folder (e.g. from the earlier Cherrapunji-only run), this notebook will merge it in. If not, it just builds fresh.

**Output:** `/content/drive/MyDrive/LITHOS/Phase9_data/lithos_all_ne_slope_units_final.gpkg`


In [ ]:
# ── STEP 1: Install dependencies ─────────────────────────────────────────────
!pip install earthengine-api geemap whitebox geopandas rasterio shapely scipy -q
print('✅ Dependencies installed')

In [ ]:
# ── STEP 2: Mount Drive + authenticate Earth Engine ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import ee

# ⚠️ EDIT THIS — your own GEE Cloud project id (Earth Engine > Register/Manage projects)
GEE_PROJECT = 'your-gee-project-id'

try:
    ee.Initialize(project=GEE_PROJECT)
    print('✅ Earth Engine initialized (existing credentials)')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('✅ Earth Engine authenticated + initialized')

In [ ]:
# ── STEP 3: Define the 8 NE state bounding boxes ─────────────────────────────
# Format: (south, west, north, east)
# Overall coverage target: 21.5N-29.5N, 87.8E-97.4E
NE_REGIONS = {
    'arunachal_pradesh': (26.5, 91.5, 29.5, 97.4),   # includes Mechuka, Tawang, Anini
    'assam':              (24.0, 89.5, 28.0, 96.5),
    'meghalaya':          (25.0, 89.5, 26.2, 92.8),
    'nagaland':           (25.2, 93.3, 27.2, 95.3),
    'manipur':            (23.8, 93.0, 25.7, 94.8),
    'mizoram':            (21.9, 92.2, 24.5, 93.5),
    'tripura':            (22.9, 91.0, 24.5, 92.4),
    'sikkim':             (27.0, 88.0, 28.2, 88.9),
}

for name, (s, w, n, e) in NE_REGIONS.items():
    print(f'  {name:<20} S={s} W={w} N={n} E={e}')

In [ ]:
# ── STEP 4: Kick off Copernicus GLO-30 DEM exports to Drive ──────────────────
DRIVE_DEM_FOLDER = 'LITHOS_Phase9_NE_DEM'  # will appear under My Drive/

def export_dem(name, bbox):
    south, west, north, east = bbox
    region = ee.Geometry.Rectangle([west, south, east, north])
    dem = ee.Image('COPERNICUS/DEM/GLO30').select('DEM').clip(region)
    task = ee.batch.Export.image.toDrive(
        image=dem,
        description=f'lithos_ne_{name}_dem',
        folder=DRIVE_DEM_FOLDER,
        fileNamePrefix=f'{name}_dem',
        region=region,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e10
    )
    task.start()
    return task

tasks = {}
print('Starting GEE export tasks (this only queues them, does not wait)...')
for name, bbox in NE_REGIONS.items():
    tasks[name] = export_dem(name, bbox)
    print(f'  queued: {name}')

In [ ]:
# ── STEP 5: Poll until all exports finish ─────────────────────────────────────
import time

def poll_tasks(tasks, interval_s=30):
    pending = set(tasks.keys())
    while pending:
        for name in list(pending):
            state = tasks[name].status()['state']
            if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
                print(f'  {name}: {state}')
                pending.discard(name)
        if pending:
            print(f'  ...{len(pending)} still running, checking again in {interval_s}s')
            time.sleep(interval_s)
    print('✅ All export tasks finished')

poll_tasks(tasks)

In [ ]:
# ── STEP 6: Copy exported DEMs from Drive into the local workspace ───────────
import os, glob, shutil

os.makedirs('lithos_data/dem_ne', exist_ok=True)
drive_dem_path = f'/content/drive/MyDrive/{DRIVE_DEM_FOLDER}'

found = glob.glob(f'{drive_dem_path}/*.tif')
print(f'Found {len(found)} DEM tif(s) in Drive folder {DRIVE_DEM_FOLDER}')

for f in found:
    dst = f'lithos_data/dem_ne/{os.path.basename(f)}'
    shutil.copy(f, dst)
    size = os.path.getsize(dst) // 1024
    print(f'  {os.path.basename(dst):<35} {size:>7} KB')

In [ ]:
# ── STEP 7: Set up WhiteboxTools ──────────────────────────────────────────────
import whitebox

wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(os.path.abspath('lithos_data/dem_ne'))
wbt.verbose = False
print('✅ WhiteboxTools ready:', wbt.version().splitlines()[0])

In [ ]:
# ── STEP 8: Hillslope (slope-unit) segmentation per region ───────────────────
# Pipeline per DEM: fill depressions -> D8 pointer -> flow accumulation ->
# extract streams (threshold) -> Hillslopes (splits the DEM into left/right
# hillslope polygons draining to each stream segment — this IS the slope-unit
# equivalent WhiteboxTools ships with).

def segment_hillslopes(region_name):
    dem_file = f'{region_name}_dem.tif'
    if not os.path.exists(f'lithos_data/dem_ne/{dem_file}'):
        print(f'  {region_name}: DEM missing, skip')
        return None

    filled   = f'{region_name}_filled.tif'
    pointer  = f'{region_name}_d8ptr.tif'
    accum    = f'{region_name}_accum.tif'
    streams  = f'{region_name}_streams.tif'
    slope    = f'{region_name}_slope.tif'
    hillslope= f'{region_name}_hillslopes.tif'

    wbt.fill_depressions(dem_file, filled)
    wbt.d8_pointer(filled, pointer)
    wbt.d8_flow_accumulation(filled, accum, out_type='cells')
    # Stream threshold: cells with accumulation > 200 (same convention as Phase 9 Cherrapunji run)
    wbt.extract_streams(accum, streams, threshold=200.0)
    wbt.hillslopes(pointer, streams, hillslope)
    wbt.slope(filled, slope, units='degrees')

    return {
        'dem': f'lithos_data/dem_ne/{filled}',
        'hillslope': f'lithos_data/dem_ne/{hillslope}',
        'slope': f'lithos_data/dem_ne/{slope}',
    }

hillslope_outputs = {}
for region in NE_REGIONS:
    print(f'--- {region} ---')
    result = segment_hillslopes(region)
    if result:
        hillslope_outputs[region] = result
        print(f'  done: {region}')

In [ ]:
# ── STEP 9: Convert hillslope rasters to polygons + attach slope/elevation ───
import numpy as np
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd

def hillslopes_to_gdf(region_name, paths, min_pixels=4, min_slope_deg=8):
    with rasterio.open(paths['hillslope']) as src:
        hs_arr    = src.read(1)
        transform = src.transform
        nodata    = src.nodata

    with rasterio.open(paths['slope']) as src:
        slope_arr = src.read(1).astype(float)

    with rasterio.open(paths['dem']) as src:
        dem_arr = src.read(1).astype(float)
        dem_arr[dem_arr < -9000] = np.nan

    valid_ids = np.unique(hs_arr)
    valid_ids = valid_ids[(valid_ids != nodata) & (valid_ids > 0)] if nodata is not None else valid_ids[valid_ids > 0]

    polygons, unit_ids, mean_slopes, mean_elevs = [], [], [], []

    for uid in valid_ids:
        mask = (hs_arr == uid)
        if mask.sum() < min_pixels:
            continue
        mask_u8 = mask.astype(np.uint8)
        for geom, val in shapes(mask_u8, transform=transform):
            if val == 1:
                poly = shape(geom)
                if not poly.is_valid or poly.area < 0.0001:
                    continue
                polygons.append(poly)
                unit_ids.append(int(uid))
                mean_slopes.append(float(np.nanmean(slope_arr[mask])))
                mean_elevs.append(float(np.nanmean(dem_arr[mask])))
                break

    if not polygons:
        return None

    gdf = gpd.GeoDataFrame({
        'unit_id':       unit_ids,
        'region':        region_name,
        'slope_degrees': mean_slopes,
        'elevation_m':   mean_elevs,
        'center_lat':    [p.centroid.y for p in polygons],
        'center_lon':    [p.centroid.x for p in polygons],
        'area_km2':      [round(p.area * 111**2, 3) for p in polygons],
        'geometry':      polygons,
    }, crs='EPSG:4326')

    gdf = gdf[gdf['slope_degrees'] > min_slope_deg].reset_index(drop=True)
    return gdf

ne_gdfs = []
for region, paths in hillslope_outputs.items():
    print(f'Vectorizing {region}...')
    g = hillslopes_to_gdf(region, paths)
    if g is not None and len(g) > 0:
        ne_gdfs.append(g)
        print(f'  {len(g)} slope units')
    else:
        print(f'  no valid units (likely flat terrain)')

In [ ]:
# ── STEP 10: Soil parameters per state (extend Phase 9's Cherrapunji table) ──
SOIL_PARAMS = {
    'cherrapunji':         {'cohesion_kpa': 8.0,  'friction_angle_deg': 28.0, 'unit_weight_knm3': 18.0, 'failure_depth_m': 3.5, 'threshold_72h_mm': 150},
    'sikkim':              {'cohesion_kpa': 15.0, 'friction_angle_deg': 28.0, 'unit_weight_knm3': 18.0, 'failure_depth_m': 2.0, 'threshold_72h_mm': 110},
    'manipur':             {'cohesion_kpa': 9.0,  'friction_angle_deg': 25.0, 'unit_weight_knm3': 18.5, 'failure_depth_m': 3.0, 'threshold_72h_mm': 130},
    'arunachal_pradesh':   {'cohesion_kpa': 7.0,  'friction_angle_deg': 23.0, 'unit_weight_knm3': 17.5, 'failure_depth_m': 4.5, 'threshold_72h_mm': 100},
    'nagaland':            {'cohesion_kpa': 11.0, 'friction_angle_deg': 27.0, 'unit_weight_knm3': 18.0, 'failure_depth_m': 3.0, 'threshold_72h_mm': 140},
    'assam':               {'cohesion_kpa': 6.0,  'friction_angle_deg': 20.0, 'unit_weight_knm3': 17.0, 'failure_depth_m': 5.0, 'threshold_72h_mm':  90},
    'meghalaya':           {'cohesion_kpa': 8.0,  'friction_angle_deg': 28.0, 'unit_weight_knm3': 18.0, 'failure_depth_m': 3.5, 'threshold_72h_mm': 150},
    'mizoram':             {'cohesion_kpa': 9.5,  'friction_angle_deg': 26.0, 'unit_weight_knm3': 18.0, 'failure_depth_m': 3.5, 'threshold_72h_mm': 145},
    'tripura':             {'cohesion_kpa': 10.0, 'friction_angle_deg': 24.0, 'unit_weight_knm3': 17.5, 'failure_depth_m': 3.0, 'threshold_72h_mm': 135},
}

import math

def compute_fos(slope_deg, rain_72h, soil):
    beta = math.radians(slope_deg)
    if abs(math.sin(beta) * math.cos(beta)) < 1e-6:
        return 99.0
    m = min(1.0, rain_72h / soil['threshold_72h_mm'])
    fos = (
        (soil['cohesion_kpa'] + (soil['unit_weight_knm3'] - m*9.81) * soil['failure_depth_m'] *
         math.cos(beta)**2 * math.tan(math.radians(soil['friction_angle_deg'])))
        /
        (soil['unit_weight_knm3'] * soil['failure_depth_m'] * math.sin(beta) * math.cos(beta))
    )
    return round(max(0.1, fos), 3)

def classify_slope(d):
    if d > 45: return 'VERY STEEP'
    elif d > 30: return 'STEEP'
    elif d > 20: return 'MODERATE'
    else: return 'GENTLE'

MONSOON_RAIN = 100  # mm/72h, same convention as Phase 9 Cherrapunji run

for gdf in ne_gdfs:
    region = gdf['region'].iloc[0]
    soil = SOIL_PARAMS.get(region, SOIL_PARAMS['assam'])
    gdf['rain_72h']    = MONSOON_RAIN
    gdf['fos']         = gdf['slope_degrees'].apply(lambda s: compute_fos(s, MONSOON_RAIN, soil))
    gdf['risk_level']  = gdf['fos'].apply(lambda f: 'RED' if f < 1.0 else 'ORANGE' if f < 1.5 else 'GREEN')
    gdf['slope_class'] = gdf['slope_degrees'].apply(classify_slope)

    red    = (gdf.risk_level=='RED').sum()
    orange = (gdf.risk_level=='ORANGE').sum()
    green  = (gdf.risk_level=='GREEN').sum()
    print(f'{region:<20} units={len(gdf):<5} RED={red:<4} ORANGE={orange:<4} GREEN={green}')

In [ ]:
# ── STEP 11: Merge with the baseline file (if present) + export final gpkg ───
import geopandas as gpd

BASELINE_PATH = '/content/drive/MyDrive/LITHOS/Phase9_data/lithos_all_slope_units_final.gpkg'
DRIVE_OUT_DIR = '/content/drive/MyDrive/LITHOS/Phase9_data'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

all_parts = []
if os.path.exists(BASELINE_PATH):
    baseline = gpd.read_file(BASELINE_PATH)
    print(f'Loaded baseline: {len(baseline)} units, regions={sorted(baseline.region.unique())}')
    all_parts.append(baseline)
else:
    print('No baseline file found — building NE-only dataset')

all_parts.extend(ne_gdfs)

master = gpd.pd.concat(all_parts, ignore_index=True)
# De-duplicate in case regions overlap between baseline and new run
master = master.drop_duplicates(subset=['region', 'unit_id', 'center_lat', 'center_lon']).reset_index(drop=True)

OUT_PATH = f'{DRIVE_OUT_DIR}/lithos_all_ne_slope_units_final.gpkg'
master.to_file(OUT_PATH, driver='GPKG')

print(f'\n=== FINAL NE-WIDE DATASET ===')
print(f'Total units: {len(master):,}')
print('\nPer region:')
print(master.groupby('region')['unit_id'].count().to_string())
print('\nRisk distribution:')
print(master['risk_level'].value_counts().to_string())
print(f'\n✅ Saved: {OUT_PATH}')